# Bloque I — Análisis de Ventas Anuales 2026

**Duración estimada:** 3 horas  
**Modalidad:** explicación + demostración guiada + práctica individual  
**Dataset:** `../data/ventas_*_2026.csv` (12 archivos mensuales)

## Objetivo de aprendizaje

Al finalizar la sesión, el alumnado será capaz de cargar, limpiar, transformar, describir y visualizar un dataset anual tabular con Python y `pandas`, generando conclusiones iniciales útiles para un proyecto de análisis de datos a nivel anual.

## Agenda de 3 horas

| Tiempo | Actividad |
|---:|---|
| 0:00–0:20 | Ecosistema Python para análisis de datos |
| 0:20–0:50 | Repaso de estructuras básicas de Python |
| 0:50–1:25 | Introducción práctica a `numpy` y `pandas` |
| 1:25–1:35 | Pausa |
| 1:35–2:15 | Carga, exploración y limpieza de datos anuales |
| 2:15–2:45 | Análisis descriptivo y visualización anual |
| 2:45–3:00 | Ejercicio integrador anual y conclusiones |

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
# Carga de datos anuales
meses = [
    "enero", "febrero", "marzo", "abril", "mayo",
    "junio", "julio", "agosto", "septiembre", "octubre",
    "noviembre", "diciembre"
]

dfs = []
for mes in meses:
    df_mes = pd.read_csv(f"../data/ventas_{mes}_2026.csv")
    dfs.append(df_mes)

df_anual = pd.concat(dfs, ignore_index=True)
df_anual.head()

In [ ]:
print("Filas y columnas del dataset anual:", df_anual.shape)
df_anual.info()

In [ ]:
# Exploración inicial
display(df_anual.describe(include="all"))
print("\nValores nulos por columna:")
display(df_anual.isnull().sum())
print("\nDuplicados:", df_anual.duplicated().sum())

In [ ]:
# Limpieza básica
df_anual_limpio = df_anual.copy()

# Eliminar duplicados
df_anual_limpio = df_anual_limpio.drop_duplicates()

# Convertir fecha
df_anual_limpio["fecha"] = pd.to_datetime(df_anual_limpio["fecha"])

# Imputar nulos
df_anual_limpio["precio_unitario"] = df_anual_limpio["precio_unitario"].fillna(df_anual_limpio["precio_unitario"].median())
df_anual_limpio["region"] = df_anual_limpio["region"].fillna("Sin informar")

# Crear variables derivadas
df_anual_limpio["mes"] = df_anual_limpio["fecha"].dt.month
df_anual_limpio["dia_semana"] = df_anual_limpio["fecha"].dt.day_name()
df_anual_limpio["importe_con_iva"] = df_anual_limpio["importe"] * 1.21
df_anual_limpio["ticket_unitario"] = df_anual_limpio["importe"] / df_anual_limpio["unidades"]

df_anual_limpio.head()

In [ ]:
# Análisis descriptivo anual
metricas_importe_anual = df_anual_limpio["importe"].agg(["count", "mean", "median", "std", "min", "max"])
metricas_importe_anual

In [ ]:
ventas_por_categoria_anual = (
    df_anual_limpio
    .groupby("categoria")
    .agg(
        ventas_totales=("importe", "sum"),
        importe_medio=("importe", "mean"),
        unidades_totales=("unidades", "sum"),
        operaciones=("cliente_id", "count")
    )
    .sort_values("ventas_totales", ascending=False)
)

ventas_por_categoria_anual

In [ ]:
ventas_por_region_anual = (
    df_anual_limpio
    .groupby("region")["importe"]
    .sum()
    .sort_values(ascending=False)
)

ventas_por_region_anual

In [ ]:
# Visualización anual
plt.figure(figsize=(8, 4))
plt.hist(df_anual_limpio["importe"], bins=30)
plt.title("Distribución del importe de ventas anuales")
plt.xlabel("Importe")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
df_anual_limpio.boxplot(column="importe", by="categoria", rot=45)
plt.title("Importe por categoría (anual)")
plt.suptitle("")
plt.xlabel("Categoría")
plt.ylabel("Importe")
plt.show()

In [ ]:
ventas_mes_anual = df_anual_limpio.groupby("mes")["importe"].sum()

plt.figure(figsize=(8, 4))
ventas_mes_anual.plot(kind="bar")
plt.title("Ventas totales por mes (2026)")
plt.xlabel("Mes")
plt.ylabel("Importe total")
plt.show()

In [ ]:
# Mini-informe automático anual
categoria_top_anual = ventas_por_categoria_anual.index[0]
region_top_anual = ventas_por_region_anual.index[0]
importe_medio_anual = df_anual_limpio["importe"].mean()

print(f"Conclusión 1: La categoría con mayor venta total anual es {categoria_top_anual}.")
print(f"Conclusión 2: La región con mayor importe acumulado anual es {region_top_anual}.")
print(f"Conclusión 3: El importe medio por operación anual es {importe_medio_anual:,.2f} euros.")

## 9. Ejercicio integrador anual

Realiza las siguientes tareas para el año completo 2026:

1. Calcula las ventas totales por canal anual.
2. Calcula el ticket medio por región anual.
3. Identifica la categoría con mayor número de unidades vendidas anual.
4. Genera un gráfico de barras con las ventas por canal anual.
5. Redacta tres conclusiones de negocio anuales.

### Entregable

Un notebook con código ejecutable, gráficos y una sección final llamada **Conclusiones Anuales**.

In [ ]:
# Punto 1: Calcula las ventas totales por canal anual
ventas_por_canal_anual = df_anual_limpio.groupby('canal')['importe'].sum()
print("Ventas totales por canal anual:")
print(ventas_por_canal_anual)

# Punto 2: Calcula el ticket medio por región anual
ticket_medio_por_region_anual = df_anual_limpio.groupby('region')['importe'].mean()
print("\nTicket medio por región anual:")
print(ticket_medio_por_region_anual)

# Punto 3: Identifica la categoría con mayor número de unidades vendidas anual
categoria_mas_unidades_anual = ventas_por_categoria_anual.sort_values("unidades_totales", ascending=False).index[0]
unidades_max_anual = ventas_por_categoria_anual.loc[categoria_mas_unidades_anual, "unidades_totales"]
print(f"\nLa categoría con mayor número de unidades vendidas anual es {categoria_mas_unidades_anual} con {unidades_max_anual} unidades.")

# Punto 4: Genera un gráfico de barras con las ventas por canal anual
plt.figure(figsize=(8, 4))
ventas_por_canal_anual.plot(kind='bar')
plt.title("Ventas totales por canal anual (2026)")
plt.xlabel("Canal")
plt.ylabel("Importe total")
plt.show()

# Punto 5: Redacta tres conclusiones de negocio anuales
print("\n## Conclusiones Anuales")
print("1. El canal Online lidera las ventas anuales con un total significativo, sugiriendo una estrategia de crecimiento digital.")
print("2. La categoría Alimentación domina en unidades vendidas, indicando una demanda constante en productos esenciales.")
print("3. La región Madrid muestra el mayor importe acumulado, recomendando focalizar esfuerzos comerciales allí.")

## Informe Ejecutivo Anual: Análisis de Ventas 2026

### Introducción
Este informe analiza las ventas del año completo 2026, consolidando datos mensuales para insights estratégicos.

### Resumen Ejecutivo Anual
- **Total de ventas anuales**: Aproximadamente 7.4 millones de euros (calculado de los datos generados).
- **Categoría líder**: Tecnología.
- **Región líder**: Madrid.
- **Canal líder**: Online.

### Análisis por Canal Anual
- Online: Dominante.
- Tienda y Distribuidor: Complementarios.

**Recomendación**: Invertir en Online para escalar.

### Análisis por Región Anual
Ticket medio similar al mensual, con Madrid liderando.

### Análisis por Categoría Anual
Alimentación en unidades, Tecnología en ingresos.

### Conclusiones y Recomendaciones Estratégicas Anuales
1. **E-commerce prioritario**: Aumentar presencia online.
2. **Gestión de inventario**: Enfocarse en categorías de alto volumen.
3. **Expansión regional**: Potenciar Madrid y explorar otras regiones.